# 02 · Predict a review's rating (regression)

**Use case:** the support team wants a predicted star rating for every review as it arrives, from its text, its summary and the product it is about — a number, not a class.

**Model / lane:** relational GNN — the default GraphSAGE architecture (Langsat labels it *Baseline*), text embeddings on `review_text`, `summary`, `product.title` / `description`

**Sub-tasks**
1. Create a `data_science` project on the same three tables
2. Define the task in plain English — Langsat writes the task config (entity, target, text columns)
3. Train a relational GNN with text embeddings (the estimate is shown first)
4. Read the test metrics: MAE, RMSE, R², Spearman — against an always-the-mean baseline
5. Score one review, then look at feature importance

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
p = get_or_create_project(ls, "rating-regression", kind="data_science")
before = credits_used(ls)

reusing project 8847cd90-8444-4934-aad9-8faa372216bf (amazon-reviews-rating-regression, status=ready)
project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']


## Define + train

The task is a sentence. `train_or_reuse` prints the config Langsat derived (entity table, target column, which text columns get embedded), the training estimate, then polls the run. A finished model on the project is reused on re-runs.

In [3]:
model = train_or_reuse(p, "Predict the rating a review gives, from the review text, its summary and the product it is about",
                       task_type="supervised", subtask_type="regression", enable_text_embedding=True)
metrics = model["metrics"]
show(metrics)

reusing model bac91893-e681-4f20-9e90-70478ccd1874 (Baseline, trained 2026-09-15T09:02:45.102808+00:00)
  r2                     0.4379
  mae                    0.4967
  rmse                   0.6894
  mae_zero               4.463
  spearman               0.5317
  mae_median             0.537
  lift_at_1pct           1.1203
  lift_at_5pct           1.1091
  lift_at_10pct          1.1069
  lift_at_0.1pct         1.1203
  lift_base_rate         4.463


## Baseline

Always predicting the mean rating: what a model must beat.

In [4]:
review = p.table("review").to_pandas()
mean = review["rating"].mean()
baseline_mae = (review["rating"] - mean).abs().mean()
baseline_rmse = ((review["rating"] - mean) ** 2).mean() ** 0.5
print(f"always-mean baseline · MAE {baseline_mae:.3f} · RMSE {baseline_rmse:.3f} · R² 0.000")
print(f"model (test split)    · MAE {metrics.get('mae'):.3f} · RMSE {metrics.get('rmse'):.3f} · R² {metrics.get('r2'):.3f} · Spearman {metrics.get('spearman', float('nan')):.3f}")

always-mean baseline · MAE 0.770 · RMSE 0.980 · R² 0.000
model (test split)    · MAE 0.497 · RMSE 0.689 · R² 0.438 · Spearman 0.532


On 10,000 heavily 5★-skewed reviews the *ranking* metric is the one to read: a Spearman of ~0.5 means the model orders reviews from worst to best far better than chance, even when R² against the raw star value stays near zero. Larger samples of the same dataset move R² up.

## Score reviews

`predict.top` ranks every review by its predicted rating (from the stored predictions); `predict.predict` scores one entity on demand (50 credits).

In [5]:
top = ls.predict.top(model["model_id"], n=5)
print("ranked by:", top.get("ranked_by"), "· total ranked:", top.get("total_ranked"))
for r in top["ranking"]:
    print(f"  #{r['rank']} entity {r['entity_id']} · score {r.get('score')}")

ranked by: score · total ranked: 2000
  #1 entity 3653 · score 4.99892520904541
  #2 entity 510 · score 4.996159553527832
  #3 entity 1044 · score 4.992633819580078
  #4 entity 411 · score 4.983786582946777
  #5 entity 4791 · score 4.980241775512695


In [6]:
one = ls.predict.predict(model_id=model["model_id"], entity_id=top["ranking"][0]["entity_id"])
print("predicted rating:", one["result"]["prediction"], "· task:", one["task_type"], "· model:", one["version_label"])

predicted rating: 4.998925372715753 · task: regression · model: v1


In [7]:
imp = p.models.importance(model["model_id"])
items = imp.get("feature_importance") or imp.get("features") or imp
pd_items = items[:10] if isinstance(items, list) else list(items.items())[:10]
for row in pd_items:
    print(" ", row)

  ['verified', 100.0]
  ['review_text', 20.99391382593831]
  ['summary', 12.089802282614553]


In [8]:
charged = credits_used(ls) - before
save_metrics(".", {"notebook": "02_rating_regression", "task": "regression · review.rating", "model": model.get("model_type"),
                   "project_id": p.id, "model_id": model["model_id"], "training_duration_sec": model.get("training_duration_sec"),
                   "credits_charged_this_run": charged,
                   "headline": {"mae": metrics.get("mae"), "rmse": metrics.get("rmse"), "r2": metrics.get("r2"), "spearman": metrics.get("spearman")},
                   "baseline": {"mae": round(baseline_mae, 4), "rmse": round(baseline_rmse, 4), "r2": 0.0}})

wrote results/metrics.json


PosixPath('results/metrics.json')